# Outlier: Minimal Causal Reconstruction Validation

This notebook is the second validation layer for the Chapter 3 Outlier result.

The previous test asked:

> Given our existing causal graph, does `c2` satisfy a strict reproduction criterion with exact copies and causally independent offspring?

It did.

This notebook asks the more fundamental question:

> Does that result survive when we change the **cell-level causal reconstruction** itself?

The current repository tracer uses a local *but-for* test: remove one live predecessor; if the child would turn off, record that predecessor as causal.

Hintze & Bohm (2026) describe a redundancy-aware procedure that identifies a minimal causal subset of live predecessors and, where several minimal causal sets exist, conservatively includes all contributing clusters.

The notebook therefore does four things:

1. reconstructs the Outlier rule and checks all 512 neighbourhoods;
2. compares the current but-for selector with an exhaustive minimal-live-subset selector;
3. rebuilds the full 512 × 512, 1,600-generation causal graph under both selectors in one simulation;
4. reruns the exact-copy / independent-offspring `c2` reproduction test on the rebuilt graph.

**Important scientific boundary:** the phrase *minimal causal subset* in the paper is not fully specified by the prose alone. The implementation here is explicit and falsifiable: it uses every **inclusion-minimal live-only sufficient submask** of the observed live neighbourhood and unions contributors if several such sets exist. The paper reports that no multiple minimal causal sets were found in Outlier. We therefore treat that statement as a validation target. If this notebook finds such cases in neighbourhoods actually used by the run, we have discovered a mismatch that must be resolved against the published code rather than explained away.


In [ ]:
from pathlib import Path
import importlib.util
import json
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SCRIPT = ROOT / "scripts" / "books" / "digital-life" / "test_outlier_minimal_causal_lineage.py"
LINEAGE = ROOT / "scripts" / "books" / "digital-life" / "ch10_outlier_lineage.py"
REPORT = ROOT / "research" / "digital-life" / "ch03-outlier-minimal-causality.json"

print("ROOT   :", ROOT)
print("SCRIPT :", SCRIPT)
print("LINEAGE:", LINEAGE)


## Load the validation module

The notebook uses the exact same implementation as the command-line test. This prevents the notebook and the test script from quietly drifting into two different experiments.


In [ ]:
spec = importlib.util.spec_from_file_location("minimal_validation", SCRIPT)
validation = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = validation
spec.loader.exec_module(validation)

outlier = validation.load_module(LINEAGE)
rule = outlier.RULE

print("rule entries:", len(rule))
print("live outputs:", int(rule.sum()))
assert len(rule) == 512
assert int(rule.sum()) == 220


## 1. Truth-table causal diagnostic

For each of the 220 neighbourhood states that produce a live child:

- `but_for_mask` contains each live predecessor whose individual removal kills the child;
- `minimal_sets` contains every inclusion-minimal live-only subset that is itself sufficient to produce the child;
- `minimal_union` conservatively includes every live cell appearing in at least one minimal set.

This is cheap: the complete causal ambiguity of the local rule can be inspected before running a single large simulation.


In [ ]:
minimal_sets, minimal_union, but_for = validation.build_causal_lookup(rule)
truth = validation.truth_table_diagnostics(
    rule, minimal_sets, minimal_union, but_for
)

summary = {
    k: v
    for k, v in truth.items()
    if k not in {"examples", "multiple_minimal_codes"}
}
summary


In [ ]:
# Show the first diagnostic disagreements.
truth["examples"][:10]


### Interpretation gate

The paper states that its validation encountered no multiple minimal causal sets in Outlier.

If `multiple_minimal_set_codes > 0`, that does **not** mean the paper is wrong. It means our operational interpretation of *minimal causal set* is not yet proven to be identical to theirs.

The next question is more relevant still: are any such neighbourhoods actually encountered in our canonical trajectory?


## 2. Canonical dual reconstruction

This is the expensive cell.

It runs the CA **once** and builds two causal graphs in parallel:

- the repository's current single-removal / but-for graph;
- the exhaustive minimal-live-subset graph.

The structural trajectory is identical in both cases. Only the causal edge selector differs.

Expected structural sanity values from the earlier canonical experiment:

- 138,891 clusters
- 144 rotation-equivalent `c2` occurrences


In [ ]:
SIZE = 512
GENERATIONS = 1600
MIN_DT = 100
MAX_DT = 900

_, c2_bitmap = outlier.derive_c2_signature(SIZE)

(
    clusters,
    minimal_edges,
    but_for_edges,
    clusters_by_time,
    encountered_codes,
) = validation.run_dual_experiment(
    outlier=outlier,
    size=SIZE,
    generations=GENERATIONS,
    minimal_union=minimal_union,
    but_for=but_for,
)

rot_signature, _ = outlier.derive_c2_signature(SIZE)
rot_occurrences = outlier.c2_occurrences(clusters, rot_signature)

{
    "clusters": len(clusters),
    "minimal_edges": len(minimal_edges),
    "but_for_edges": len(but_for_edges),
    "rotation_equivalent_c2": len(rot_occurrences),
}


## 3. Did the run actually use ambiguous local neighbourhoods?

This is the key diagnostic for matching the published causal implementation.

A rule-table ambiguity that never occurs in the experiment cannot affect this run. An ambiguity used thousands of times can.


In [ ]:
multiple_codes = set(truth["multiple_minimal_codes"])
encountered_multiple = sorted(
    c for c in multiple_codes if encountered_codes.get(c, 0) > 0
)

{
    "distinct_live_output_codes_encountered": len(encountered_codes),
    "multiple_minimal_codes_encountered": len(encountered_multiple),
    "total_uses": sum(encountered_codes[c] for c in encountered_multiple),
    "first_codes": encountered_multiple[:20],
}


## 4. Compare the two causal graphs

The CA trajectory has not changed. Any edge difference below is entirely due to the causal definition.


In [ ]:
old_set = validation.edge_set(but_for_edges)
min_set = validation.edge_set(minimal_edges)

graph_comparison = {
    "but_for_edges": len(old_set),
    "minimal_set_edges": len(min_set),
    "shared_edges": len(old_set & min_set),
    "minimal_only_edges": len(min_set - old_set),
    "but_for_only_edges": len(old_set - min_set),
}
graph_comparison


## 5. Re-run the strict reproduction criterion

Identity is now strict:

- translation is allowed because cluster bitmaps are cropped;
- rotation is **not** normalized;
- offspring must be causal first returns to the exact `c2` bitmap;
- the parent must have at least two offspring;
- at least one offspring pair must be causally independent of one another.

This is the same lineage criterion used in the preceding validation, now evaluated on the rebuilt minimal-set causal graph.


In [ ]:
exact_minimal, events_minimal = validation.strict_replication_events(
    clusters=clusters,
    edges=minimal_edges,
    c2_bitmap=c2_bitmap,
    outlier=outlier,
    min_dt=MIN_DT,
    max_dt=MAX_DT,
)

exact_old, events_old = validation.strict_replication_events(
    clusters=clusters,
    edges=but_for_edges,
    c2_bitmap=c2_bitmap,
    outlier=outlier,
    min_dt=MIN_DT,
    max_dt=MAX_DT,
)

{
    "exact_c2_occurrences": len(exact_minimal),
    "qualifying_parents_minimal_graph": len(events_minimal),
    "qualifying_parents_but_for_graph": len(events_old),
    "minimal_graph_pass": bool(events_minimal),
    "but_for_graph_pass": bool(events_old),
}


In [ ]:
# Inspect the earliest qualifying parent under the minimal-set graph.
events_minimal[0] if events_minimal else None


## 6. Decision

There are two independent questions.

### A. Does reproduction survive the stronger causal graph?

If `minimal_graph_pass == True`, then the exact-copy, independent-offspring result survives this causal reconstruction.

### B. Have we reproduced Hintze & Bohm's causal implementation?

Only if the local-method diagnostics are also consistent with the published method. In particular, the paper reports no multiple minimal causal sets in Outlier. If this implementation encounters them, the correct conclusion is:

> reproduction survives **our explicit minimal-subset reconstruction**, but this reconstruction is not yet demonstrated to be identical to the published implementation.

That is still useful science. It tells us the result is robust to a substantial change in causal attribution while identifying the precise remaining replication gap.


In [ ]:
decision = {
    "strict_reproduction_survives_minimal_graph": bool(events_minimal),
    "multiple_minimal_codes_encountered": len(encountered_multiple),
    "paper_method_match_not_established": bool(encountered_multiple),
}

if decision["strict_reproduction_survives_minimal_graph"]:
    print("LINEAGE RESULT: PASS")
else:
    print("LINEAGE RESULT: FAIL")

if decision["paper_method_match_not_established"]:
    print(
        "METHOD STATUS: DIAGNOSTIC MISMATCH — compare the causal selector "
        "against the published Zenodo code before calling this a reproduction "
        "of Hintze & Bohm's implementation."
    )
else:
    print(
        "METHOD STATUS: no multiple-minimal-set conflict encountered in this run."
    )

decision


## 7. Write a compact experimental record

This record is intended to be cited by the manuscript/appendix. It contains the causal definition as well as the result, so the number cannot become detached from the method that generated it.


In [ ]:
report = {
    "configuration": {
        "size": SIZE,
        "generations": GENERATIONS,
        "min_dt": MIN_DT,
        "max_dt": MAX_DT,
    },
    "truth_table": summary,
    "encountered_multiple_minimal_codes": encountered_multiple,
    "encountered_multiple_minimal_uses": int(
        sum(encountered_codes[c] for c in encountered_multiple)
    ),
    "graph_comparison": graph_comparison,
    "strict_exact_c2": {
        "exact_occurrences": len(exact_minimal),
        "qualifying_parents_minimal_graph": len(events_minimal),
        "qualifying_parents_but_for_graph": len(events_old),
        "minimal_graph_pass": bool(events_minimal),
        "events": events_minimal,
    },
    "decision": decision,
    "method_note": (
        "Minimal graph uses the union of all inclusion-minimal live-only "
        "sufficient neighbourhood submasks. This is an explicit validation "
        "implementation, not yet claimed to be byte-for-byte equivalent to "
        "the published Hintze-Bohm causal code."
    ),
}

REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(REPORT)


## Manuscript consequence

Use the following hierarchy when updating Chapter 3:

**If the minimal graph fails:**  
Our own 1,600-generation run supports branching reproduction only under the simpler but-for graph. Keep the published result as the strong reproduction evidence and weaken the book's own-run claim.

**If the minimal graph passes but the method diagnostic conflicts with the paper:**  
Say that the reproduction result survives a second, redundancy-aware causal reconstruction, but do **not** say we reproduced Hintze & Bohm's causal algorithm. Resolve the implementation difference first.

**If the minimal graph passes and the published-code comparison later matches:**  
The full chain is earned:

`exact recurrence + published-style causal ancestry + branching + independent offspring → reproduction survives the test`.
